In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("Medallion_Example").getOrCreate()
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]
columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
Raw_data = spark.createDataFrame(data,columns)


In [0]:
Raw_data.write.format("delta").mode("overwrite").save("/Volumes/medallion_sample/bronze/raw_data")

In [0]:
bronze_data=spark.read.format("delta").load("/Volumes/medallion_sample/bronze/raw_data")
bronze_data.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
silver_data=spark.read.format("delta").load("/Volumes/medallion_sample/bronze/raw_data")
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
cleaned_data=silver_data.dropDuplicates()

In [0]:
silver_data.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

In [0]:
cleaned_data.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|   

In [0]:
cleaned_data=cleaned_data.withColumn("date", col("date").cast("date"))
cleaned_data=cleaned_data.withColumn("amount", col("amount").cast("int"))

In [0]:
cleaned_data.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- amount: integer (nullable = true)
 |-- quantity: long (nullable = true)



In [0]:
window = Window.partitionBy("customer_id","product").orderBy(col("date").desc())

df = cleaned_data.withColumn("rn", row_number().over(window)) \
              .filter(col("rn") == 1) \
              .drop("rn")
              
df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



In [0]:
df=df.filter(col("amount")>0)
df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



In [0]:
df=df.fillna({
    "city":"unknown"
})
df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|  unknown|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



In [0]:
df.write.format("delta").mode("overwrite").save("/Volumes/medallion_sample/gold/cleaned_data")

In [0]:
gold_df=spark.read.format("delta").load("/Volumes/medallion_sample/gold/cleaned_data")
gold_df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|  unknown|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



##Tasks

Aggregations
Total sales by product
Total sales by category
Total sales by city

Customer Metrics
Total orders per customer
Total spending per customer

Top Analysis
Top selling product
Top customer


In [0]:
df.select("product","amount","category").groupBy("product").agg(sum("amount").alias("Total Sales")).show()

+----------+-----------+
|   product|Total Sales|
+----------+-----------+
|    Laptop|     105000|
|    Tablet|      22000|
|    Mobile|      33000|
|     Watch|       8000|
|Headphones|       3000|
+----------+-----------+



In [0]:
df.select("product","amount","category").groupBy("category").agg(sum("amount").alias("Total Sales")).show()

+-----------+-----------+
|   category|Total Sales|
+-----------+-----------+
|Electronics|     160000|
|Accessories|      11000|
+-----------+-----------+



In [0]:
df.select("product","amount","city").groupBy("city").agg(sum("amount").alias("Total Sales")).show()

+---------+-----------+
|     city|Total Sales|
+---------+-----------+
|Hyderabad|      72000|
|  Chennai|      33000|
|    Delhi|      55000|
|   Mumbai|       8000|
|     NULL|       3000|
+---------+-----------+



In [0]:
df.select("customer_id","order_id").groupBy("customer_id").count().show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C001|    2|
|       C002|    1|
|       C003|    1|
|       C004|    1|
|       C005|    1|
|       C007|    1|
+-----------+-----+



In [0]:
df.groupBy("customer_id").agg(sum(col("amount") * col("quantity")).alias("Total Spending")).show()

+-----------+--------------+
|customer_id|Total Spending|
+-----------+--------------+
|       C001|         72000|
|       C002|         18000|
|       C003|         55000|
|       C004|          8000|
|       C005|          3000|
|       C007|         30000|
+-----------+--------------+



In [0]:
#top selling customer
df.orderBy("quantity").groupBy("product").count().show(1)

+-------+-----+
|product|count|
+-------+-----+
| Laptop|    2|
+-------+-----+
only showing top 1 row


In [0]:
#Top Customer
df.groupBy("customer_id").agg(sum("amount")).orderBy(desc("sum(amount)")).show(1)

+-----------+-----------+
|customer_id|sum(amount)|
+-----------+-----------+
|       C001|      72000|
+-----------+-----------+
only showing top 1 row
